In [1]:
from pils.loader.stout import StoutLoader
from pils.loader.path import PathLoader
from pils.flight import Flight
from pils.sensors.camera import Camera, PhotogrammetryConfig

%matplotlib widget
import matplotlib.pyplot as plt


## 1 — Load flight metadata

Use `StoutLoader` if you have a Stout inventory, otherwise `PathLoader` to scan a directory.

The `flight_info` dict must contain at minimum:
- `drone_data_folder_path`
- `aux_data_folder_path`
- `proc_data_folder_path` — where `proc/photogrammetry/` will be created

In [2]:
# --- Option A: Stout inventory ---
# stout = StoutLoader()
# flight_meta = stout.load_single_flight(flight_name="flight_20251206_1304")

#--- Option B: directory scan ---
loader = PathLoader("/data/POLOCALC/")
flights = loader.load_single_flight(flight_name="flight_20251206_1530")
flight_meta = flights

flight_meta

2026-06-19 15:32:37,894 - pils.loader.path - INFO - Loading single flight: flight_id=None, flight_name=flight_20251206_1530
2026-06-19 15:32:37,895 - pils.loader.path - INFO - Loading all flights from all campaigns...
2026-06-19 15:32:37,896 - pils.loader.path - WARNING - Could not build flight dict for calibration  /data/POLOCALC/campaigns/202412/20241215/calibration : time data 'tion ' does not match format '%Y%m%d_%H%M'
2026-06-19 15:32:37,897 - pils.loader.path - WARNING - Could not build flight dict for Store-V2 /data/POLOCALC/campaigns/202412/.Spotlight-V100/Store-V2: time data '2' does not match format '%Y%m%d_%H%M'
2026-06-19 15:32:37,897 - pils.loader.path - WARNING - Could not build flight dict for 24_12_17 /data/POLOCALC/campaigns/202412/calibration_PUC/24_12_17: time data '7' does not match format '%Y%m%d_%H%M'
2026-06-19 15:32:37,898 - pils.loader.path - WARNING - Could not build flight dict for 24_12_18 /data/POLOCALC/campaigns/202412/calibration_PUC/24_12_18: time data '

{'campaign_name': '202511',
 'flight_name': 'flight_20251206_1530',
 'flight_date': '20251206',
 'takeoff_datetime': '2025-12-06T15:30:00+00:00',
 'landing_datetime': '2025-12-06T15:30:00+00:00',
 'drone_data_folder_path': '/data/POLOCALC/campaigns/202511/20251206/flight_20251206_1530/drone',
 'aux_data_folder_path': '/data/POLOCALC/campaigns/202511/20251206/flight_20251206_1530/aux',
 'processed_data_folder_path': '/data/POLOCALC/campaigns/202511/20251206/flight_20251206_1530/proc'}

## 1 — Load flight metadata

Use `StoutLoader` if you have a Stout inventory, otherwise `PathLoader` to scan a directory.

In [3]:
flight = Flight(flight_meta)

# Drone telemetry — auto-detects DJI / BlackSquare
flight.add_drone_data()

# Camera — use_photogrammetry=False loads raw frames/log (Alvium or Sony)
# This stores both:
#   flight.raw_data.payload_data.camera     → pl.DataFrame (for sync)
#   flight.raw_data.payload_data.camera_obj → Camera instance (for run_photogrammetry)
flight.add_camera_data(use_photogrammetry=False)

print(flight.raw_data)

2026-06-19 15:32:42,431 - pils.flight - INFO - Drone : /data/POLOCALC/campaigns/202511/20251206/flight_20251206_1530/drone/20251206_153038_drone.dat
2026-06-19 15:32:42,431 - pils.drones.DJIDrone - INFO - PATH: /data/POLOCALC/campaigns/202511/20251206/flight_20251206_1530/drone/20251206_153038_drone.dat
2026-06-19 15:32:44,243 - pils.drones.DJIDrone - INFO - Loaded 3642 GPS messages from DAT file
2026-06-19 15:32:44,247 - pils.drones.DJIDrone - INFO - Loaded 3349 RTK messages from DAT file
2026-06-19 15:32:44,308 - pils.drones.DJIDrone - INFO - Converting timestamps to milliseconds
2026-06-19 15:32:44,429 - pils.sensors.camera - INFO - Parsing telemetry from /data/POLOCALC/campaigns/202511/20251206/flight_20251206_1530/aux/camera/20251206_153038_video.mp4 (5040.2 MB, timeout=2520s)


   ⏳ telemetry_parser running... 0s / 2520s


Process Process-1:
Traceback (most recent call last):
  File "/home/fastori/anaconda3/envs/photo/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/home/fastori/anaconda3/envs/photo/lib/python3.12/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/home/fastori/Desktop/ARS/pils/pils/sensors/camera.py", line 530, in _worker
    imu_data = parser.normalized_imu()
               ^^^^^^^^^^^^^^^^^^^^^^^
pyo3_runtime.PanicException: assertion failed: v.len() == 3
thread '<unnamed>' panicked at /home/runner/work/telemetry-parser/telemetry-parser/src/sony/mod.rs:41:9:
assertion failed: v.len() == 3
note: run with `RUST_BACKTRACE=1` environment variable to display a backtrace
2026-06-19 15:32:44,913 - pils.sensors.camera - WARNING - Sony telemetry unavailable for /data/POLOCALC/campaigns/202511/20251206/flight_20251206_1530/aux/camera/20251206_153038_video.mp4 (telemetry-parser crashed (exitcode=1) on /da

=== DRONE DATA ===
Drone:
shape: (6_971, 42)
┌────────────┬──────────┬───────────┬──────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ tick       ┆ msg_type ┆ GPS:date  ┆ GPS:time ┆ … ┆ RTK:pos_f ┆ RTK:pos_f ┆ RTK:pos_f ┆ RTK:gps_s │
│ ---        ┆ ---      ┆ ---       ┆ ---      ┆   ┆ lg_3      ┆ lg_4      ┆ lg_5      ┆ tate      │
│ i64        ┆ i64      ┆ f64       ┆ f64      ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│            ┆          ┆           ┆          ┆   ┆ f64       ┆ f64       ┆ f64       ┆ f64       │
╞════════════╪══════════╪═══════════╪══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 26096229   ┆ 2096     ┆ 0.0       ┆ 0.0      ┆ … ┆ null      ┆ null      ┆ null      ┆ null      │
│ 28031339   ┆ 2096     ┆ 2.0251206 ┆ 152727.0 ┆ … ┆ null      ┆ null      ┆ null      ┆ null      │
│            ┆          ┆ e7        ┆          ┆   ┆           ┆           ┆           ┆           │
│ 28671752   ┆ 2096     ┆ 2.0251206 ┆ 152727.0

In [15]:
camera_obj = flight.raw_data.payload_data.camera_obj
dh = camera_obj.load_data

## 2 — Build Flight and load sensors

In [ ]:
# photogrammetry_config.yaml lives in pils/config/
cfg = PhotogrammetryConfig("/home/fastori/Desktop/ARS/pils/pils/config/photogrammetryConfig.yaml")

print("Camera matrix:\n", cfg.camera_matrix)
print("Distortion coeffs:", cfg.distortion_coeffs)
print("Finder params:     ", cfg.finder)
print("PnP params:        ", cfg.pnp)
print("Drone corr params: ", cfg.drone_correlation)

## 3 — Load photogrammetry config

Edit `photogrammetry_config.yaml` with your camera calibration values before running.

## 4 — Run the photogrammetry pipeline

- `csv_file`: geodetic targets + telescope positions CSV
- `output_dir`: where intermediate `.ecsv` dictionaries are saved
- `check_results`: if set, diagnostic plots are saved here
- `start_from_dict`: resume mid-pipeline by pointing to an existing `.ecsv`

In [ ]:
# ===================================================================
# FULL PHOTOGRAMMETRY PIPELINE (Step 4)
# Runs on ALL frames, using manual corrections as anchor points
# ===================================================================

# Grab the Camera object stored by add_camera_data()
camera_obj = flight.raw_data.payload_data.camera_obj

result = camera_obj.run_photogrammetry(
    csv_file="/data/POLOCALC/campaigns/202511/metadata/202511_coordinates.csv",
    config=cfg,
    flight=flight,
    output_dir="/home/fastori/Desktop/ARS/photogrammetry_results_1225",
    check_results=None,       # None → auto-saves plots to proc/photogrammetry/plots/
    start_from_dict=None,     # e.g. "dictionary_p3.ecsv" to resume mid-pipeline
)

print(f"Result: {result.shape[0]} frames × {result.shape[1]} columns")
result.head()

## 5 — Inspect the result

### 5.1 Column overview

In [ ]:
# Group columns by pipeline step for easier orientation
groups = {
    "p2 — image targets":     [c for c in result.columns if c in ["frame", "x", "y", "target_id"]],
    "p3 — PnP attitude":      [c for c in result.columns if c.startswith(("rvec", "tvec", "quat", "proj"))],
    "p4 — drone GPS":         [c for c in result.columns if c.startswith(("drone_", "time"))],
    "p5 — GPS-corrected":     [c for c in result.columns if c.endswith("_corr") or "projection_error" in c],
    "p6 — telescope frame":   [c for c in result.columns if c in ["tel_name", "az", "el"]],
    "p7 — LOS frame":         [c for c in result.columns if "LOS" in c],
}

for group, cols in groups.items():
    if cols:
        print(f"\n{group}")
        print("  ", cols)

### 5.2 Projection error — quality check

Low projection error (< 2–3 px) means the PnP solution is reliable for that frame.

In [ ]:
pe = result.select(["frame", "time", "projection_error"]).drop_nulls()

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(pe["time"].to_numpy(), pe["projection_error"].to_numpy(), lw=1)
ax.axhline(2.0, color="orange", ls="--", label="2 px threshold")
ax.axhline(5.0, color="red",    ls="--", label="5 px threshold")
ax.set_xlabel("Time (s)")
ax.set_ylabel("Projection error (px)")
ax.set_title("PnP projection error per frame")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Median: {pe['projection_error'].median():.2f} px")
print(f"Max:    {pe['projection_error'].max():.2f} px")
print(f"Frames > 5px: {(pe['projection_error'] > 5).sum()}")

### 5.3 Attitude — roll, pitch, yaw over time

In [ ]:
import numpy as np

# Use GPS-corrected rvec if available, fall back to raw PnP
rvec_cols = ["rvec_x_corr", "rvec_y_corr", "rvec_z_corr"]
if not all(c in result.columns for c in rvec_cols):
    rvec_cols = ["rvec_x", "rvec_y", "rvec_z"]

att = result.select(["time"] + rvec_cols).drop_nulls()
t   = att["time"].to_numpy()
rv  = att.select(rvec_cols).to_numpy()  # (N, 3) Rodrigues vectors

# Convert to degrees for readability
angles_deg = np.degrees(rv)

fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
labels = ["Roll (°)", "Pitch (°)", "Yaw (°)"]
for i, (ax, label) in enumerate(zip(axes, labels)):
    ax.plot(t, angles_deg[:, i], lw=1)
    ax.set_ylabel(label)
    ax.grid(True, alpha=0.3)
axes[-1].set_xlabel("Time (s)")
axes[0].set_title("Camera attitude (GPS-corrected rvec)")
plt.tight_layout()
plt.show()

### 5.4 Drone position in ENU

In [ ]:
pos = result.select(["time", "drone_E", "drone_N", "drone_U"]).drop_nulls()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Ground track
axes[0].plot(pos["drone_E"].to_numpy(), pos["drone_N"].to_numpy(), lw=1)
axes[0].set_xlabel("East (m)")
axes[0].set_ylabel("North (m)")
axes[0].set_title("Ground track (ENU)")
axes[0].set_aspect("equal")
axes[0].grid(True, alpha=0.3)

# Altitude over time
axes[1].plot(pos["time"].to_numpy(), pos["drone_U"].to_numpy(), lw=1, color="steelblue")
axes[1].set_xlabel("Time (s)")
axes[1].set_ylabel("Up / altitude (m)")
axes[1].set_title("Altitude over time")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 5.5 LOS frame attitude (p7 output)

In [ ]:
los_cols = ["time", "yaw_LOS", "pitch_LOS", "roll_LOS"]
if all(c in result.columns for c in los_cols):
    los = result.select(los_cols).drop_nulls()
    t   = los["time"].to_numpy()

    fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
    for ax, col, label in zip(
        axes,
        ["yaw_LOS", "pitch_LOS", "roll_LOS"],
        ["Yaw LOS (°)", "Pitch LOS (°)", "Roll LOS (°)"],
    ):
        ax.plot(t, np.degrees(los[col].to_numpy()), lw=1)
        ax.set_ylabel(label)
        ax.grid(True, alpha=0.3)
    axes[-1].set_xlabel("Time (s)")
    axes[0].set_title("Drone attitude in line-of-sight frame")
    plt.tight_layout()
    plt.show()
else:
    print("LOS columns not found — pipeline may not have reached step p7.")

## 6 — Save result

Save as Parquet for fast reloading, or as CSV for external tools.

In [ ]:
# The result is already saved automatically to:
#   flight.flight_info['proc_data_folder_path'] / 'photogrammetry' / 'attitude_reconstruction.parquet'
#
# Reload it later with:
import os
proc_path = flight.flight_info["proc_data_folder_path"]
parquet_path = os.path.join(proc_path, "photogrammetry", "attitude_reconstruction.parquet")

import polars as pl
result = pl.read_parquet(parquet_path)
print(f"Loaded {result.shape[0]} frames from {parquet_path}")
result.head()

## 7 — Resume from a mid-pipeline checkpoint

If the pipeline failed or you want to re-run from step p4 onwards, point `start_from_dict` at the last good dictionary.

In [ ]:
# Resume from a mid-pipeline checkpoint — skips all steps before p4
import os
proc_path = flight.flight_info["proc_data_folder_path"]
checkpoint = os.path.join(proc_path, "photogrammetry", "dictionary_p3.ecsv")

result = camera_obj.run_photogrammetry(
    csv_file="/path/to/targets.csv",
    config=cfg,
    flight=flight,
    start_from_dict=checkpoint,
)

result.head()